# Notebook 04: Pricing Pipeline (CP-SAT -> Paired Pricing Artifact (+ Legacy WHT) -> DQI Handoff)

This notebook is the BMW-fit bridge: define/load a pricing instance, solve it exactly with CP-SAT,
and export both a Stage A paired pricing artifact and a backward-compatible legacy WHT surrogate artifact.


In [1]:
import json
import sys
from pathlib import Path

try:
    from notebooks._helpers import repo_root
except ModuleNotFoundError:
    from _helpers import repo_root

ROOT = repo_root()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
ROOT


PosixPath('/Users/zenith/Desktop/dqi-pricing-benchmark')

In [2]:

try:
    from ortools.sat.python import cp_model  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Notebook 04 requires OR-Tools (ortools). Install with: pip install ortools"
    ) from exc


In [3]:
try:
    from notebooks._helpers import decode_config_table, load_instance
except ModuleNotFoundError:
    from _helpers import decode_config_table, load_instance

from src.classical_baselines import brute_force_solver
from src.ilp_to_xorsat_exact import (
    paired_pricing_artifact_from_dict,
    paired_pricing_artifact_to_dict,
)
from src.pricing_ilp import REQUIRED_COMPARATIVE_SUBSET, solve_pricing_pair
from src.pricing_model_ortools import solve_pricing_with_cp_sat
from src.reduction import surrogate_artifact_from_dict, surrogate_artifact_to_dict


In [4]:

instance_path = ROOT / "data" / "instances" / "pricing_5feat_10bit.json"
prob = load_instance(instance_path)
instance_id = "pricing_5feat_10bit"

print(f"Loaded pricing instance: {instance_path.name}")
print(f"n_features={prob.n_features}, n_bits={prob.n_bits}, n_configs={prob.n_configurations}")


Loaded pricing instance: pricing_5feat_10bit.json
n_features=5, n_bits=10, n_configs=1024


In [5]:

cp_result = solve_pricing_with_cp_sat(prob, instance_id=instance_id)
x_bf, f_bf = brute_force_solver(prob)

cp_summary = {
    "status": cp_result["status"],
    "instance_id": cp_result["instance_id"],
    "x_opt": cp_result["x_opt"],
    "x_opt_bits": format(cp_result["x_opt"], f"0{prob.n_bits}b"),
    "objective_value_F": cp_result["objective_value"],
    "optimized_objective_value": cp_result["optimized_objective_value"],
    "bruteforce_objective_F": f_bf,
    "matches_bruteforce_objective": abs(cp_result["objective_value"] - f_bf) < 1e-9,
    "solve_time_s": cp_result["solve_time_s"],
}
cp_summary


{'status': 'optimal',
 'instance_id': 'pricing_5feat_10bit',
 'x_opt': 16,
 'x_opt_bits': '0000010000',
 'objective_value_F': 3000.0,
 'optimized_objective_value': 191.0,
 'bruteforce_objective_F': 3000.0,
 'matches_bruteforce_objective': True,
 'solve_time_s': 0.0488684969895985}

In [6]:

assert cp_result["objective_value"] == prob.evaluate(cp_result["x_opt"])
decode_config_table(prob, cp_result["x_opt"])


,feature,tier,price
0,Heated Seats,Standard,500.0
1,Panoramic Roof,Standard,800.0
2,Premium Audio,Premium,800.0
3,Adaptive Cruise,Standard,600.0
4,Ambient Lighting,Standard,300.0


In [7]:
paired = solve_pricing_pair(
    prob,
    instance_id=instance_id,
    k=15,
    source_metadata={
        "instance_id": instance_id,
        "pricing_instance_path": str(instance_path.relative_to(ROOT)),
    },
)

paired_path = ROOT / "results" / "paired_pricing" / f"{instance_id}.paired.json"
paired_path.parent.mkdir(parents=True, exist_ok=True)
paired_path.write_text(json.dumps(paired_pricing_artifact_to_dict(paired), indent=2))

legacy_path = ROOT / "data" / "instances" / f"surrogate_{instance_id}_k15.json"
legacy_wht = paired["encodings"]["wht_truncated"]
legacy_path.write_text(json.dumps(surrogate_artifact_to_dict(legacy_wht), indent=2))

print(f"Saved paired artifact to: {paired_path}")
print(f"Saved legacy WHT artifact to: {legacy_path}")
print(f"encodings={sorted(paired['encodings'].keys())}")

Saved paired artifact to: /Users/zenith/Desktop/dqi-pricing-benchmark/results/paired_pricing/pricing_5feat_10bit.paired.json
Saved legacy WHT artifact to: /Users/zenith/Desktop/dqi-pricing-benchmark/data/instances/surrogate_pricing_5feat_10bit_k15.json
encodings=['ilp_derived', 'wht_truncated']


In [8]:
payload = json.loads(paired_path.read_text())
loaded = paired_pricing_artifact_from_dict(payload)

allowed_top_level = {
    "paired_artifact_version",
    "instance_id",
    "pairing_kind",
    "source_metadata",
    "encodings",
}
assert set(loaded.keys()) == allowed_top_level
assert not ({"B", "v", "n", "m", "bit_order", "term_weights"} & set(loaded.keys()))

assert set(loaded["encodings"].keys()) == {"wht_truncated", "ilp_derived"}

for enc_name in ("wht_truncated", "ilp_derived"):
    enc = loaded["encodings"][enc_name]
    assert REQUIRED_COMPARATIVE_SUBSET.issubset(set(enc.keys()))

wht = loaded["encodings"]["wht_truncated"]
ilp = loaded["encodings"]["ilp_derived"]

# Lineage/provenance checks.
assert loaded["instance_id"] == wht["source_metadata"]["instance_id"]
assert loaded["instance_id"] == ilp["source_metadata"]["instance_id"]
assert wht["source_metadata"]["pricing_instance_path"] == ilp["source_metadata"]["pricing_instance_path"]

# Cross-encoding consistency checks.
assert int(wht["n"]) == int(ilp["n"])
assert int(wht["m"]) == int(ilp["m"])
if str(wht["bit_order"]) != str(ilp["bit_order"]):
    align = ilp["source_metadata"].get("bit_order_alignment")
    assert isinstance(align, dict), "bit_order mismatch requires source_metadata.bit_order_alignment"

# Shape integrity checks.
for enc_name, enc in (("wht_truncated", wht), ("ilp_derived", ilp)):
    n = int(enc["n"])
    m = int(enc["m"])
    assert enc["B"].shape == (m, n), f"{enc_name}: invalid B shape"
    assert enc["v"].shape == (m,), f"{enc_name}: invalid v shape"
    assert enc["term_weights"].shape == (m,), f"{enc_name}: invalid term_weights shape"

legacy_payload = json.loads(legacy_path.read_text())
legacy_loaded = surrogate_artifact_from_dict(legacy_payload)
assert legacy_loaded["B"].shape == (legacy_loaded["m"], legacy_loaded["n"])
assert legacy_loaded["v"].shape == (legacy_loaded["m"],)
assert legacy_loaded["term_weights"].shape == (legacy_loaded["m"],)

{
    "schema_ok": True,
    "paired_path": str(paired_path.relative_to(ROOT)),
    "legacy_path": str(legacy_path.relative_to(ROOT)),
    "encodings": sorted(loaded["encodings"].keys()),
    "wht_n_m": (int(wht["n"]), int(wht["m"])),
    "ilp_n_m": (int(ilp["n"]), int(ilp["m"])),
    "bit_order_status": "matched" if str(wht["bit_order"]) == str(ilp["bit_order"]) else "aligned",
}

{'schema_ok': True,
 'paired_path': 'results/paired_pricing/pricing_5feat_10bit.paired.json',
 'legacy_path': 'data/instances/surrogate_pricing_5feat_10bit_k15.json',
 'encodings': ['ilp_derived', 'wht_truncated'],
 'wht_n_m': (10, 15),
 'ilp_n_m': (10, 15),
 'bit_order_status': 'matched'}

**Handoff complete:** this notebook produced a validated paired pricing artifact and a backward-compatible
legacy WHT surrogate artifact for downstream notebook consumers.

Note: the separate 3-period pricing extension is documented in Notebook 09. This notebook remains the single-period pricing pipeline.
